# Creating your own benchmark and mechanism

This tutorial builds a complete benchmark and a custom mechanism from a mathematical definition, and then runs them through the same optimizers as the fishery benchmark. It is an ordinary Jupyter notebook: every code cell runs on the present API, from the first cell to the last, and the whole notebook executes in well under two minutes. The last section starts a tiny bilevel run through Ray, so the notebook needs the project environment (`uv sync`).

The tutorial begins with the mathematical problem definition and only then maps the model to the code abstractions. The goal is a new benchmark that reuses the mechanism algorithms and the bilevel optimization infrastructure unchanged.

# 1. Start from the mathematical model

Before creating classes, define the multi-agent controlled dynamical system:

\[
\mathcal E
=
(
\mathcal S,
\{\mathcal A_i\}_{i=1}^{N},
P,
\{R_i\}_{i=1}^{N},
\{\mathcal O_i\}_{i=1}^{N},
\gamma
).
\]

You need to specify:

1. state \(s_t\in\mathcal S\);
2. each agent action \(a_{i,t}\);
3. transition \(P(s_{t+1}\mid s_t,a_t)\);
4. intrinsic/base reward \(R_i\);
5. observation function \(O_i\);
6. initial-state/reset distribution;
7. horizon/termination semantics.

Only after the benchmark is mathematically clear should a mechanism be introduced.

# 2. Separate benchmark dynamics from regulation

The benchmark answers:

> What happens in the world when delivered actions are executed?

The mechanism answers:

> What actions reach the world, what reward reaches the learner, and what information reaches the learner?

Mechanism intervention:

\[
o_t^\*
=
\mathcal M_\theta^O(s_t,o_t),
\]

\[
a_t^\*
=
\mathcal M_\theta^A(s_t,a_t),
\]

\[
r_t^\*
=
\mathcal M_\theta^R(
r_t,
s_t,
a_t^\*,
s_{t+1}
).
\]

The transition must consume the regulated action:

\[
s_{t+1}
\sim
P(\cdot\mid s_t,a_t^\*).
\]

Keeping this boundary clean is what makes one quota algorithm reusable in a fishery, irrigation system, energy market, traffic network, or another benchmark.

In the code, the three interventions are methods of one `Mechanism` class. Its `decode` method turns the raw action that reached the mechanism into the value that is delivered. Its `apply` method returns a *residual*: a small `MDPState` that the agent owning the mechanism adds to the shared state, and that can lower an action or shape a reward. Its `observe` method returns the residual of the observations.

Two details differ from the formulas. A reward residual is computed before the transition, so in the code it can read the state \(s_t\) and the delivered action, but not \(s_{t+1}\). The observations are built after the transition, from the new state. Section 4 gives the order of a step, and Section 11 explains each method.

# 3. Ingredients of a custom benchmark

A benchmark is a handful of small objects, and each one answers one of the questions of Section 1. The table lists where each piece of the mathematical model lives in the code.

| Mathematical piece | Code object |
|---|---|
| State and constants | Entries of the shared `MDPState`: `state` holds the stocks that change, `params` holds the constants. The reset hook writes both. |
| Agent action | One `Mechanism` per action component of the agent. Its `decode` method maps the raw policy output to the delivered value. The `action_space` of the mechanism is the space the policy outputs. |
| Transition | The one method marked `@transition` in a subclass of `MultiAgentEnv`. |
| Base reward and observation | The `reward` and `observation` methods of an `Agent` subclass. They are overridden by name and are not hooks. |
| Initial state and horizon | The one method marked `@reset`, and the `horizon` argument of the environment. |
| Metrics | A `MetricSchema` subclass, written to by the environment's logger. |

The regulation side adds three more objects. The regulator is an `AgentConfig` whose mechanisms are `MechanismConfig` objects, either shipped ones such as `Quota` or your own. A `RegulatorEnv` subclass turns the metrics of the inner learning level into one fitness per candidate mechanism. A `ContextSchema` payload carries that fitness back to the World.

The next cell gathers every import of the notebook. The `core` package is importable because the project is installed in the environment (`uv sync`), whatever the directory of the notebook, so no setup cell is needed. The `examples` package is not installed, and this notebook does not use it.

In [ ]:
from __future__ import annotations

import dataclasses
import logging
import tempfile
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import ClassVar, Optional
from unittest import mock

import numpy as np
import ray
from gymnasium import spaces
from pydantic import Field

from core.adaptors.ray.marl_env import RLlibMultiAgentEnvAdapter
from core.agents.base import Agent, AgentConfig
from core.callbacks import log_and_report_episode_metrics, tag_episode_with_env_idx
from core.envs.hooks import reset, transition
from core.envs.marl_regulated import MultiAgentEnv
from core.envs.regulator import RegulatorEnv
from core.envs.schema import EpisodeRolloutSchema
from core.mechanism.algorithms.penalty import ThresholdPenalty
from core.mechanism.algorithms.quota import Quota
from core.mechanism.algorithms.social_influence import SocialInfluence
from core.mechanism.base import MDPState, Mechanism
from core.mechanism.config import MechanismConfig
from core.metrics.enums import ReduceProtocol
from core.metrics.logger import MetricLogger
from core.optimizers.appo.config import APPOptimizerConfig
from core.optimizers.bilevel import BilevelConfig
from core.optimizers.es.config import ESConfig
from core.optimizers.es.optimizer import ESOptimizer
from core.reporting.csv import CSVConfig
from core.utils import ACTION_TEMPERATURE, sigmoid, smooth_positive_zero_at_origin
from core.world.context import Context, ContextSchema, MechanismContext, MechanismStatus

# 4. Hooks

The environment hooks mark the two benchmark-specific functions that no mechanism can supply. There are exactly two of them:

- `@reset` marks the method that draws the initial state and returns it as an `MDPState` residual;
- `@transition` marks the method that advances the shared state by one step.

The base class finds the marked methods when your subclass is created (in `__init_subclass__`), so there is nothing to register. A class body may mark each hook at most once.

Observation and reward are **not** hooks. A follower computes them in the `observation` and `reward` methods of its `Agent` subclass, which you override by name (Section 7.2 does it).

One `step` of the environment runs in this order:

1. every agent in turn, the leaders first, applies its mechanisms to the actions (`Agent.action`) and then adds its reward (`Agent.reward`). A leader's residuals on the followers' actions and rewards are therefore already in the state when a follower decodes its own action and computes its own reward;
2. the `@transition` method advances the state, and the termination flags are set;
3. the metric logger records the step: the transition has pushed its own fields, and the environment pushes the generic reward fields itself (`reward_total`, `reward_mean`, and so on);
4. the observations of the new state are built: the followers' `observation` methods, plus the `observe` residuals of the leaders' mechanisms.

The `@reset` method runs after the environment has fetched the candidate mechanism from the World and written it as the leaders' action, so the reset itself may already depend on the candidate. The next cell checks the rule that a hook may be marked once.

In [ ]:
try:

    class TwoResets(MultiAgentEnv):
        @reset
        def first_reset(self, mdp):
            return MDPState()

        @reset
        def second_reset(self, mdp):
            return MDPState()

except TypeError as error:
    print("TypeError:", error)
else:
    raise AssertionError("a class that marks two reset hooks must be rejected")

# 5. Schemas and runtime context

It is useful to distinguish two meanings of "schema".

## 5.1 Runtime context

The runtime context is what travels between the World and the environments, and the regulator's candidate mechanism is its main content. A `MechanismContext` carries:

- `index`: the position of the candidate in the population;
- `seed`: the training seed this copy of the candidate is meant for, because the regulator publishes one copy per training seed, and `eval_seed`, an optional environment seed for evaluation that the current regulators leave at `None`;
- `status`: the lifecycle of the candidate in the World, which is `published` when the regulator appends it, then `train` or `eval` once an environment of that mode has fetched it, and `done` when the regulator publishes its fitness;
- `env_id`: the environment that produced the context, which is `None` while the regulator publishes it;
- `mechanism`: for each mechanism of the leader, the action the optimizer chose, as an array;
- `metrics`: filled when the candidate has been scored, by the `reward` method of your regulator environment (Section 23).

At reset, the environment asks the World for the candidate and writes it as the leaders' action. In production the World is a Ray actor. For hand stepping, the next cell defines a stand-in that answers with a scripted candidate, and a context manager that makes `ray.get` return its argument.

In [ ]:
class ScriptedWorld:
    """Stand-in for the Ray World actor: each reset gets the next scripted candidate."""

    def __init__(self, answers=()):
        self.answers = list(answers)
        self.get_mechanism_by_id = SimpleNamespace(remote=self._fetch)

    def _fetch(self, **kwargs):
        return self.answers.pop(0) if self.answers else None


def candidate(**mechanism_actions):
    """Candidate context that hands the given mechanism actions to the leaders."""
    return MechanismContext(
        index=0,
        env_id=None,
        seed=0,
        status=MechanismStatus.train,
        mechanism=mechanism_actions,
        metrics=None,
    )


def stand_in_ray_get():
    """Make `ray.get` return its argument, as the stand-in World returns values."""
    return mock.patch.object(ray, "get", lambda ref, *args, **kwargs: ref)


# A candidate as a training environment receives it from the World.
candidate(quota=np.array([0.82], dtype=np.float32))

## 5.2 Metrics schema

Important benchmark quantities should also be represented in a `MetricSchema`. The runtime context answers:

> What happened on this step?

The metrics schema answers:

> How should this quantity accumulate, reduce, and be queried for reporting?

Keep those responsibilities separate. Each field declares how the per-step values reduce to one value per episode. The schema below extends the generic episode schema with three quantities of the common-pool benchmark of Section 6: the mean normalized stock, the normalized stock at the end of the episode, and the total harvest that was actually delivered. The environment pushes one value per step with `logger.push`, and `logger.reduce` returns the episode values. The generic fields of the episode schema, such as `reward_mean`, are pushed by the environment itself. The regulator of Section 23 reads these fields back.

In [ ]:
class CommonPoolMetricSchema(EpisodeRolloutSchema):
    """Metrics of the common-pool benchmark, one value per step, reduced per episode."""

    stock_norm_mean: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )
    stock_norm_last: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.LAST}
    )
    realized_harvest_total: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.SUM}
    )


metric_logger = MetricLogger.from_schema(CommonPoolMetricSchema)
for stock_norm, harvest in ((0.80, 30.0), (0.82, 25.0), (0.83, 20.0)):
    metric_logger.push(key=("stock_norm_mean",), value=stock_norm)
    metric_logger.push(key=("stock_norm_last",), value=stock_norm)
    metric_logger.push(key=("realized_harvest_total",), value=harvest)

episode = metric_logger.reduce()
assert np.isclose(episode.stock_norm_mean, (0.80 + 0.82 + 0.83) / 3)
assert np.isclose(episode.stock_norm_last, 0.83)
assert np.isclose(episode.realized_harvest_total, 75.0)
(episode.stock_norm_mean, episode.stock_norm_last, episode.realized_harvest_total)

# 6. Worked benchmark: renewable common-pool resource

We create a minimal renewable resource.

State:

\[
x_t\in[0,K].
\]

Normalized state:

\[
\bar x_t=\frac{x_t}{K}.
\]

Each agent chooses one extraction fraction:

\[
a_{i,t}
=
\begin{bmatrix}
u_{i,t}
\end{bmatrix},
\qquad
u_{i,t}\in[0,1].
\]

Logistic growth:

\[
G(x_t)
=
r x_t
\left(
1-\frac{x_t}{K}
\right).
\]

Maximum extraction capacity for agent \(i\):

\[
H_i^{\max}.
\]

Attempted extraction:

\[
H_{i,t}=u_{i,t}H_i^{\max}.
\]

Transition. The harvest that is actually delivered cannot exceed what the resource holds after growth, so the transition first limits the attempted total:

\[
H_t^{\mathrm{real}}
=
\min\left(\sum_iH_{i,t},\;x_t+G(x_t)\right),
\qquad
x_{t+1}
=
\operatorname{clip}
\left(
x_t+G(x_t)-H_t^{\mathrm{real}},
0,
K
\right).
\]

Base reward, paid on the attempted extraction after any regulation, as a dimensionless fraction:

\[
r_{i,t}=u_{i,t}.
\]

Like the fishery benchmark, which pays the harvest fraction, it pays the amount that was asked for and not the amount that the resource could deliver. Paying the fraction keeps the base reward in \([0,1]\), the same units as the tax of Section 10.

Observation:

\[
o_{i,t}=[\bar x_t].
\]

In the code the observation has one more entry, which stays at zero until a mechanism fills it (Sections 10 and 20).

The logistic growth law is Verhulst's (1838), and Schaefer (1954) introduced it as the surplus-production model of a fishery, which is the form used here.

*References.* Verhulst, P.-F. (1838). Notice sur la loi que la population poursuit dans son accroissement. *Correspondance mathématique et physique*, 10, 113-121. Schaefer, M. B. (1954). Some aspects of the dynamics of populations important to the management of the commercial marine fisheries. *Bulletin of the Inter-American Tropical Tuna Commission*, 1(2), 27-56 (reprinted in *Bulletin of Mathematical Biology*, 53, 253-279, 1991, doi:10.1007/BF02464432).

# 7. Implement the benchmark

The benchmark is four small pieces, written bottom-up: a pure transition function, an `Agent` subclass for the base reward and the observation, a follower mechanism that fixes the action semantics, and the environment class with its two hooks.

## 7.1 The transition as a pure function

Keep the mathematics in a function that knows nothing about agents, optimizers or Ray. Section 15 tests it before any learning starts. The function returns the next stock and the harvest that was delivered.

In [ ]:
def logistic_step(
    x: float, harvest: float, *, r: float, K: float
) -> tuple[float, float]:
    """Advance the stock by one step of logistic growth minus the delivered harvest.

    Parameters
    ----------
    x : float
        Stock at the start of the step, in resource units, in [0, K].
    harvest : float
        Total attempted extraction of the step, in resource units, >= 0.
    r : float
        Intrinsic growth rate per step.
    K : float
        Carrying capacity, in resource units.

    Returns
    -------
    tuple of float
        The stock at the end of the step, in [0, K], and the harvest delivered,
        which never exceeds the stock available after growth.
    """
    growth = r * x * (1.0 - x / K)
    available = max(x + growth, 0.0)
    delivered = min(harvest, available)
    return float(np.clip(available - delivered, 0.0, K)), float(delivered)


logistic_step(800.0, 50.0, r=0.3, K=1_000.0)

## 7.2 The agent: base reward and observation

An `Agent` subclass defines the intrinsic reward and the observation of a follower by overriding two methods by name. Both read the shared `MDPState` and return a residual that carries only their own entries.

Two details come from the state conventions. The state, the actions and the rewards are indexed by time, so `mdp.state[key][mdp.t]` is the value at the current step. The observation has two entries: the normalized stock, and a slot that is reserved for the observation of a mechanism (the scarcity tax of Section 10 fills it). Declaring the slot in the observation space from the start is the rule of Section 20.

The reward reads the **decoded** extraction fraction from `mdp.actions`: by the time the reward is computed, the follower's mechanism (Section 7.3) has decoded the raw action and any leader mechanism has already acted on it.

In [ ]:
STOCK = "stock"  # key of the resource in the state
HARVEST = "harvest"  # id of the follower's extraction mechanism
FOLLOWER = "harvester"  # prefix of the follower ids: "harvester:0", "harvester:1", ...
NUM_HARVESTERS = 2

# Raw policy output of the extraction mechanism, and the observation of a harvester
# (normalized stock, then one slot reserved for a mechanism observation).
RAW_ACTION = spaces.Box(-np.inf, np.inf, shape=(1,), dtype=np.float32)
OBSERVATION = spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
# Searched coordinate of a regulator mechanism: one scalar in [0, 1].
UNIT = spaces.Box(0.0, 1.0, shape=(1,), dtype=np.float32)


class Harvester(Agent):
    """Follower that extracts from the common pool."""

    def observation(self, mdp: MDPState) -> MDPState:
        x_norm = mdp.state[STOCK][mdp.t] / mdp.params["K"]
        return MDPState(obs={self.id: np.asarray([x_norm, 0.0], dtype=np.float32)})

    def reward(self, mdp: MDPState) -> MDPState:
        fraction = float(mdp.actions[self.id][HARVEST][mdp.t])
        return MDPState(rewards={self.id: fraction})


class HarvesterConfig(AgentConfig):
    """Configuration of a `Harvester`; `count` copies get the ids "harvester:<i>"."""

    agent_cls: ClassVar[type[Agent]] = Harvester

## 7.3 The follower mechanism: action semantics

The policy of a follower outputs a raw, unbounded number \(z\). A mechanism of the follower turns it into the extraction fraction, so the meaning of the action lives in one named place. The `Extraction` mechanism decodes

\[
u=\operatorname{sigmoid}\!\left(\frac{z}{T}\right),
\qquad
T=4,
\]

which keeps \(u\) in \((0,1)\) whatever the policy outputs. The temperature \(T=4\) is the constant `ACTION_TEMPERATURE` of `core.utils`. It is a heuristic scale and not a value taken from the literature. It lives in one constant because the mechanisms of a leader, which read the raw output, must use the same value as the follower that decodes it. It changes nothing else, so `apply` returns an empty residual. Every mechanism comes with a `MechanismConfig`: a frozen dataclass that names the mechanism class and builds it for an agent.

In [ ]:
class Extraction(Mechanism):
    """Decode the raw policy output into an extraction fraction in (0, 1)."""

    def decode(self, mdp: MDPState, action) -> float:
        z = float(np.asarray(action, dtype=np.float32).reshape(-1)[0])
        return sigmoid(z / ACTION_TEMPERATURE)

    def apply(self, mdp: MDPState, action: float) -> MDPState:
        return MDPState()


class ExtractionConfig(MechanismConfig):
    mechanism_cls: ClassVar[type[Mechanism]] = Extraction


harvester = HarvesterConfig(
    id=FOLLOWER,
    policy_id="harvester_policy",
    mechanisms=(ExtractionConfig(id=HARVEST, action_space=RAW_ACTION),),
    observation_space=OBSERVATION,
)

## 7.4 The environment: reset and transition hooks

Reset. By default the initial stock is \(x_0=0.8K\), so the first tests are exact. A positive `x_init_sigma` draws it instead, \(x_0=\operatorname{clip}(0.8K(1+\sigma\xi),0,K)\) with \(\xi\sim\mathcal N(0,1)\), from the environment's own seeded generator `self.rng`. This gives the seeded-repeat test of Section 22 a randomness to seed.

The reset hook returns a residual with the stock under `state` and the constants under `params`. The constants travel in the state so that generic mechanisms can read them: the `Quota` of Section 9 divides the stock by `mdp.params["K"]`.

Transition. The transition receives the state **after** the leaders' mechanisms have acted. The harvest of each follower is therefore already the regulated one, read from `mdp.actions`, and the hook never has to know that a regulator exists. It advances the state with `mdp.advance`, which appends one step, and it logs the metrics of the step.

In [ ]:
class CommonPoolEnv(MultiAgentEnv):
    """Renewable common-pool resource shared by the follower agents."""

    def __init__(self, *, ecology_cfg, **kwargs):
        super().__init__(**kwargs)
        self.r = float(ecology_cfg.get("r", 0.3))
        self.K = float(ecology_cfg.get("K", 1_000.0))
        self.x_init = float(ecology_cfg.get("x_init", 0.8 * self.K))
        self.x_init_sigma = float(ecology_cfg.get("x_init_sigma", 0.0))
        self.H_max = float(ecology_cfg.get("max_harvest_per_agent", 0.02 * self.K))
        self.ecology = {"r": self.r, "K": self.K, "H_max": self.H_max}

    @reset
    def reset_resource(self, mdp: MDPState) -> MDPState:
        x0 = self.x_init
        if self.x_init_sigma > 0.0:
            x0 *= 1.0 + self.x_init_sigma * self.rng.normal()
        x0 = float(np.clip(x0, 0.0, self.K))
        return MDPState(state={STOCK: x0}, params=self.ecology)

    @transition
    def renewable_transition(self, mdp: MDPState) -> MDPState:
        t = mdp.t
        x = float(mdp.state[STOCK][t])
        attempted = sum(
            float(mdp.actions[aid][HARVEST][t]) * self.H_max for aid in self.followers
        )
        x_next, delivered = logistic_step(x, attempted, r=self.r, K=self.K)
        if self.logger is not None:
            self.logger.push(key=("stock_norm_mean",), value=x_next / self.K)
            self.logger.push(key=("stock_norm_last",), value=x_next / self.K)
            self.logger.push(key=("realized_harvest_total",), value=delivered)
        return mdp.advance(state={STOCK: x_next})

## 7.5 Build the environment, reset it, and step it by hand

The environment takes the World, the follower configurations, the leaders' configurations, the horizon, a seed, the metric schema, and a `mechanism_id`, the index of the candidate this environment evaluates. At reset the environment asks the World for that candidate, and the stand-in World of Section 5.1 ignores the index. The follower ids must be of the form `"<agent>:<i>"`, because a leader's mechanism finds its targets by that prefix. The first environment has no leader at all, so it is the benchmark alone.

A step is driven the way the RLlib adapter drives it: write the raw actions into the state with `mdp.update(actions=...)`, then call `env.step(mdp)`. The raw action \(z=4\ln 3\) requests the fraction \(u=\operatorname{sigmoid}(\ln 3)=0.75\). Two harvesters with \(H^{\max}=0.02K=20\) then attempt \(2\times 0.75\times 20=30\) units from a stock of 800. The stock then becomes \(800+48-30=818\), and each harvester is paid the fraction 0.75.

In [ ]:
followers = {
    f"{FOLLOWER}:{i}": dataclasses.replace(harvester, id=f"{FOLLOWER}:{i}")
    for i in range(NUM_HARVESTERS)
}


def build_env(candidate_context=None, leaders=None, *, horizon=5, seed=0, ecology=None):
    """Build the benchmark with a scripted World that may hold one candidate."""
    return CommonPoolEnv(
        world=ScriptedWorld([candidate_context] if candidate_context else []),
        mechanism_id=0,
        horizon=horizon,
        seed=seed,
        agents_cfg_dict=followers,
        leaders_cfg_dict={} if leaders is None else leaders,
        schema=CommonPoolMetricSchema,
        ecology_cfg=ecology or {},
    )


def raw_actions(z):
    """The same raw extraction output z for every follower, as the policies emit it."""
    return {aid: {HARVEST: np.array([z], dtype=np.float32)} for aid in followers}


z_request = ACTION_TEMPERATURE * np.log(3.0)  # requests the fraction 0.75

with stand_in_ray_get():
    env = build_env()
    mdp = env.reset(MDPState(aids=set(followers)))
    assert mdp.state[STOCK][0] == 800.0
    assert np.allclose(mdp.obs["harvester:0"][0], [0.8, 0.0])

    mdp.update(actions=raw_actions(z_request))
    mdp = env.step(mdp)

assert mdp.t == 1
assert np.isclose(mdp.state[STOCK][1], 818.0)  # 800 + 0.3 * 800 * 0.2 - 30
assert np.isclose(mdp.actions["harvester:0"][HARVEST][0], 0.75)  # decoded fraction
assert np.isclose(mdp.rewards["harvester:0"][0], 0.75)  # base reward
assert np.allclose(mdp.obs["harvester:0"][1], [0.818, 0.0])
assert np.isclose(env.logger.peek().realized_harvest_total[-1], 30.0)
mdp.state[STOCK], mdp.rewards["harvester:0"]

# 8. Define action semantics explicitly

Document every action component. In this framework an agent does not own one anonymous action vector. Its action is a dictionary with one entry per mechanism, keyed by the mechanism `id`, and each entry has the `action_space` of that mechanism. For the example:

- follower, entry `"harvest"`: one raw, unbounded number, decoded to the extraction fraction.

If the benchmark later becomes richer, each new component becomes a new mechanism with its own id and its own space: a `"restore"` entry for restoration effort, an `"invest"` entry for investment, and so on. A leader's mechanism then addresses its target by name, with `acts_on=("harvester", "harvest")`. This is the documented contract that replaces reading the meaning of an action from its position in an array: a mechanism never guesses which component it is looking at, because it names the entry.

The RLlib adapter publishes the contract as the `action_spaces` and `observation_spaces` of the environment. The next cell prints and checks it.

In [ ]:
with stand_in_ray_get():
    adapter = RLlibMultiAgentEnvAdapter(build_env())

assert set(adapter.action_spaces["harvester:0"].spaces) == {HARVEST}
assert adapter.action_spaces["harvester:0"][HARVEST].shape == (1,)
adapter.action_spaces["harvester:0"]

# 9. Add an existing quota mechanism

The shipped `Quota` curtails the extraction of the targeted agents according to the state of the resource. The regulator searches one number \(q\in[0,1]\). With \(\bar x\) the normalized stock, the fraction of the effort that the quota allows is a smooth function of \(\bar x-q\): close to 0 when the stock is well below \(q\), close to 1 when it is well above. A follower that requests the fraction \(u\) is delivered about \(\min(u,\text{allowed})\). The formula is in the module docstring of `core/mechanism/algorithms/quota.py`. The logistic smoothing of the allowance, and the softened excess over it, are numerical choices of the shipped mechanism, not a published model: the docstring of the module says so, and the widths are fixed fields of `Quota` (`quota_transition_width=0.03`, `usage_transition_width=0.005`).

The wiring is entirely in the configuration:

- `acts_on=("harvester", "harvest")` names the targets: the agents whose id starts with `"harvester:"`, and their mechanism `"harvest"`;
- `obs_map={"resource_level": "stock"}` tells the quota which state entry holds the resource. The quota divides it by `mdp.params["K"]`.

The mechanism belongs to a leader, an `AgentConfig` whose id is not a follower id. The quota acts on the **raw** request \(z\) of the followers, because the leader acts first, and it applies the sigmoid itself.

The cell sets \(q=0.82\) with a stock at \(\bar x=0.8\), so the quota is binding, and lets the followers request \(u=0.75\).

In [ ]:
quota = Quota(
    id="quota",
    action_space=UNIT,
    acts_on=(FOLLOWER, HARVEST),
    obs_map={"resource_level": STOCK},
)
regulator = AgentConfig(
    id="regulator", policy_id="regulator_policy", mechanisms=(quota,)
)


def run_step(leaders, candidate_actions, z, *, horizon=5):
    """Reset with a scripted candidate, then step once with the raw output z."""
    context = candidate(**candidate_actions) if candidate_actions else None
    with stand_in_ray_get():
        env = build_env(context, leaders, horizon=horizon)
        mdp = env.reset(MDPState(aids=set(followers)))
        mdp.update(actions=raw_actions(z))
        return env, env.step(mdp)


def allowed_fraction(level, q, width=0.03):
    """Allowed fraction of the effort of the quota: the formula of its docstring."""
    return (sigmoid((level - q) / width) - sigmoid(-q / width)) / (
        sigmoid((1.0 - q) / width) - sigmoid(-q / width)
    )


def expected_delivered(requested, allowed, width=0.005):
    """Delivered fraction: the request minus its smooth excess over the allowance."""
    excess = smooth_positive_zero_at_origin(requested - allowed, width)
    return float(np.clip(requested - excess, 1e-6, 1.0 - 1e-6))


candidate_quota = {"quota": np.array([0.82], dtype=np.float32)}
env_quota, step_quota = run_step({"regulator": regulator}, candidate_quota, z_request)

# Hand value: sigmoid(-2/3) / sigmoid(6) is close to 0.34.
allowed = allowed_fraction(0.8, 0.82)
assert np.isclose(allowed, 0.34, atol=1e-3)
assert np.isclose(step_quota.state["allowed_frac"][0], allowed)

# The delivered fraction and the base reward follow from the formulas, not from the env.
delivered = step_quota.actions["harvester:0"][HARVEST][0]
expected = expected_delivered(0.75, allowed)
assert delivered < 0.75  # the quota cut the request of 0.75
assert np.isclose(delivered, expected, rtol=1e-4)
assert np.isclose(step_quota.rewards["harvester:0"][0], expected, rtol=1e-4)
print(f"requested 0.75, allowed {allowed:.4f}, delivered {delivered:.4f}")

This is the abstraction boundary:

- The quota says: "I require a resource level, which I read from the state entry that `obs_map` names, and the carrying capacity `K` from the constants."
- The benchmark says: "For me, the resource level is the entry `stock`, and `K` is in `params`."

The quota therefore does not need to know whether the resource is fish, groundwater, a reservoir, or another stock.

# 10. Create a custom mechanism from mathematics

We create a scarcity-weighted extraction tax.

## 10.1 Mathematical definition

Let:

\[
\bar x_t=x_t/K
\]

and let the regulator search a rate

\[
\lambda\in[0,1].
\]

For extraction fraction \(u_{i,t}\):

\[
r_{i,t}^{*}
=
r_{i,t}
-
\lambda
(1-\bar x_t)
u_{i,t}.
\]

The tax has three properties. A high resource level gives a small tax, and a low level gives a larger one. A harvester that does not extract pays nothing. The mechanism does not change the physical dynamics, it only shapes the reward.

The tax is a reward residual, so it fits the `apply` method. It also exposes its rate to the harvesters, through the reserved observation slot, which fits the `observe` method.

## 10.2 Implement the custom mechanism

A custom mechanism subclasses `Mechanism` and overrides three methods, and its configuration subclasses `MechanismConfig`.

The `decode(mdp, action)` method maps the coordinate that the optimizer searched, a number in \([0,1]\), to the semantic rate. It must clip, and it must not modify the state. It always receives the raw action as it arrived, never a value that was already decoded, so it does not have to be idempotent.

The `apply(mdp, rate)` method receives the decoded rate and returns the reward residual, which is \(-\lambda(1-\bar x_t)u_{i,t}\) for every follower. It reads the stock through `obs_map["resource_level"]`, the capacity from `mdp.params["K"]`, and the targets from `acts_on`. It reads the follower's request from the follower's entry in `mdp.actions`. Because the leader acts first, that entry still holds the raw output \(z\) of the policy, so the request is \(u=\operatorname{sigmoid}(z/T)\). Reading the request and not the delivered action is what makes the tax price the *request*, which Section 13 checks.

The `observe(mdp)` method returns the observation residual that writes the rate into slot 1 of every follower's observation. The environment calls it at reset and after every transition.

A fixed parameter, one that the optimizer does not search, is a keyword-only field of the configuration: here `max_rate`, the largest rate the regulator may set. The constructor validates it.

In [ ]:
class ScarcityTaxMechanism(Mechanism):
    """Tax the extraction of the targeted agents, more heavily when the stock is low.

    Parameters
    ----------
    max_rate : float
        Largest rate lambda the regulator may set, in (0, 1]. The searched
        coordinate in [0, 1] is scaled to [0, max_rate].
    """

    def __init__(self, *, max_rate: float = 1.0, **kwargs):
        super().__init__(**kwargs)
        self.max_rate = max_rate
        if not 0.0 < self.max_rate <= 1.0:
            raise ValueError("max_rate must be in (0, 1].")

    def decode(self, mdp: MDPState, action) -> float:
        coordinate = float(np.asarray(action, dtype=np.float32).reshape(-1)[0])
        return self.max_rate * float(np.clip(coordinate, 0.0, 1.0))

    def apply(self, mdp: MDPState, action: float) -> MDPState:
        if self.acts_on is None:
            raise ValueError("ScarcityTaxMechanism requires `acts_on`.")
        if self.obs_map is None or "resource_level" not in self.obs_map:
            raise ValueError("ScarcityTaxMechanism requires obs_map['resource_level'].")
        target_agent, target_mechanism = self.acts_on
        level = mdp.state[self.obs_map["resource_level"]][mdp.t] / mdp.params["K"]
        scarcity = 1.0 - level
        rewards = {}
        for aid, actions in (mdp.actions.data or {}).items():
            if not str(aid).startswith(f"{target_agent}:"):
                continue
            if target_mechanism not in actions:
                continue
            z = float(np.asarray(actions[target_mechanism][mdp.t]).reshape(-1)[0])
            requested_fraction = sigmoid(z / ACTION_TEMPERATURE)
            rewards[aid] = -action * scarcity * requested_fraction
        return MDPState(rewards=rewards)

    def observe(self, mdp: MDPState) -> MDPState:
        # The candidate arrives as the leader's raw action: decode it to read the rate.
        history = mdp.raw_actions.data.get(self.aid, {}).get(self.id)
        if not history:
            return MDPState()
        rate = self.decode(mdp, history[min(mdp.t, len(history) - 1)])
        target_agent = self.acts_on[0]
        return MDPState(
            obs={
                aid: np.asarray([0.0, rate], dtype=np.float32)
                for aid in mdp.aids
                if str(aid).startswith(f"{target_agent}:")
            }
        )


@dataclass(frozen=True, kw_only=True)
class ScarcityTax(MechanismConfig):
    """Configuration of a `ScarcityTaxMechanism`; `max_rate` is a fixed parameter."""

    mechanism_cls: ClassVar[type[Mechanism]] = ScarcityTaxMechanism
    max_rate: float = 1.0


tax = ScarcityTax(
    id="scarcity_tax",
    action_space=UNIT,
    acts_on=(FOLLOWER, HARVEST),
    obs_map={"resource_level": STOCK},
)
tax_mechanism = tax.build("regulator")
assert isinstance(tax_mechanism, ScarcityTaxMechanism)
assert tax_mechanism.max_rate == 1.0

# 11. Why each mechanism method exists

`action_space`
: The Gymnasium space of the mechanism's own action. For a regulator mechanism it is the **search space**: ES requires a `Box` with bounds \([0,1]\), and its shape is the number of dimensions the optimizer controls. For a follower mechanism it is the space the policy outputs.

`decode(mdp, action)`
: Raw action, as it arrived, to semantic value. It is the identity by default. When the mechanism is called, the decoded value is written back in place into the actions of the state, so that later readers (the reward, the transition) see the delivered value, and then `apply` receives it. `decode` always receives the raw action as it arrived, so an override does not have to be idempotent. It is where the bounds are enforced and where an optimizer coordinate in \([0,1]\) becomes a physical quantity, and it must not modify the state. The optimizer vector is always the normalized coordinate, and `decode` is the only map from it to the semantics.

`apply(mdp, action)`
: The intervention. It receives the decoded value and returns a residual `MDPState` with the changes it makes (to actions, rewards, state entries). It never edits the state itself. The agent adds the residuals of all its mechanisms.

`observe(mdp)`
: The observation residual. It is how a mechanism exposes its own state to the agents (the tax rate here, the peers' actions for `SocialInfluence`). It is separate from the optimizer coordinate because what an agent should see is not always what the optimizer searched.

`default`
: The action the mechanism takes before the World has delivered a candidate, so that an environment also runs on its own.

`acts_on` and `obs_map`
: The wiring to the benchmark (Section 14).

`MechanismConfig.build(aid)`
: A frozen configuration builds the mechanism for the agent that owns it. The configuration is what the optimizers carry and copy; fixed parameters such as `max_rate` are its keyword-only fields.

On the optimizer side, `ESOptimizer` derives the search from the action spaces of the leader's mechanisms. Its `dimension` is the total size of the spaces, and its `parameter_names` lists the mechanism ids, in the alphabetical order that Gymnasium gives the entries of a `Dict` space.

# 12. Fixed versus optimized mechanisms

A fixed mechanism searches no parameter. Its `action_space` is an empty `Box` of shape \((0,)\), so its dimension is

\[
d=0.
\]

It still takes part in every step, and the optimizer hands it an empty array. The shipped `ThresholdPenalty` is such a rule: it sanctions agents while the stock is below a threshold, with no searched parameter. An optimized mechanism has

\[
d>0.
\]

Composition adds dimensions:

\[
d_{\text{composition}}
=
\sum_i d_i.
\]

The next cell builds the optimizer of the ES regulator without starting Ray, which is enough to read the search space: the quota and the tax are searched (\(d=1+1\)), and the penalty is fixed (\(d=0\)), so it adds no name. A leader with the penalty alone gives a search of dimension 0, which the optimizer treats as a fixed mechanism.

In [ ]:
penalty = ThresholdPenalty(
    id="penalty", acts_on=(FOLLOWER, HARVEST), obs_map={"resource_level": STOCK}
)
assert penalty.action_space.shape == (0,)  # fixed rule: no searched dimension


def search_space(*mechanisms):
    """ES optimizer of one leader, built without Ray, to read its search space."""
    leader = AgentConfig(
        id="regulator", policy_id="regulator_policy", mechanisms=mechanisms
    )
    config = ESConfig().training(episodes=1, sigma=0.15, mean_lr=0.1).agents(leader)
    return ESOptimizer(config.debugging(seed=0))


searched = search_space(quota, tax, penalty)
assert searched.dimension == 2
assert searched.parameter_names == ["quota", "scarcity_tax"]
assert not searched.fixed_mode

fixed = search_space(penalty)
assert fixed.dimension == 0
assert fixed.fixed_mode
print(searched.parameter_names, searched.dimension, fixed.dimension)

# 13. Compose the quota and the custom tax

Give one leader both mechanisms:

\[
\mathcal M
=
\mathcal M_{\text{tax}}\circ\mathcal M_{\text{quota}}.
\]

Composition is residual addition, not function chaining. The mechanisms of one agent are all applied to the **same input state**, and their residuals are summed by `MDPState.add`. Three consequences follow, and the cells below check each one.

1. The order of the mechanisms does not matter. The agent walks through the entries of its candidate and sums the residuals, so the order in which the mechanisms act is the order of the keys of the candidate, and reversing it must change nothing.
2. The tax sees the request, not the quota-cut action, because the quota's cut is a residual that has not been added yet. In this benchmark the tax therefore prices what a harvester asked for.
3. To make one mechanism see the effect of another, give them to **different leaders**. Leaders act in order, and a later leader reads the state that the earlier leader already changed.

The first two checks run on a leader that owns both mechanisms. With \(q=0.82\), \(\lambda=0.5\), \(\bar x=0.8\) and a request of \(u=0.75\), the tax residual is \(-0.5\times0.2\times0.75=-0.075\) per follower.

In [ ]:
candidate_both = {
    "quota": np.array([0.82], dtype=np.float32),
    "scarcity_tax": np.array([0.5], dtype=np.float32),
}
both = AgentConfig(
    id="regulator", policy_id="regulator_policy", mechanisms=(quota, tax)
)

# Same candidate, with the entries in the opposite order: the agent now walks
# through the tax first and the quota second.
candidate_reversed = dict(reversed(candidate_both.items()))
assert list(candidate_reversed) == ["scarcity_tax", "quota"]

_, step_both = run_step({"regulator": both}, candidate_both, z_request)
_, step_reversed = run_step({"regulator": both}, candidate_reversed, z_request)

# 1. The order of the mechanisms does not matter.
assert list(step_reversed.raw_actions.data["regulator"]) == ["scarcity_tax", "quota"]
assert np.isclose(
    step_both.rewards["harvester:0"][0], step_reversed.rewards["harvester:0"][0]
)
assert np.isclose(
    step_both.actions["harvester:0"][HARVEST][0],
    step_reversed.actions["harvester:0"][HARVEST][0],
)

# 2. The quota alone gave the delivered fraction; the tax adds -lambda (1 - x) u with
#    u the REQUESTED fraction 0.75, not the delivered one.
reward_quota_only = step_quota.rewards["harvester:0"][0]
assert np.isclose(step_both.actions["harvester:0"][HARVEST][0], delivered)
assert np.isclose(
    step_both.rewards["harvester:0"][0] - reward_quota_only, -0.5 * 0.2 * 0.75
)
print(
    "tax residual per follower:",
    step_both.rewards["harvester:0"][0] - reward_quota_only,
)

The third consequence needs two leaders. The tax moves to a second leader, the *tax office*, which acts after the regulator. It reads the state after the quota's cut, so it now prices the delivered fraction instead of the request: \(-0.5\times0.2\times\) delivered.

In [ ]:
tax_office = AgentConfig(id="tax_office", policy_id="tax_policy", mechanisms=(tax,))
_, step_two_leaders = run_step(
    {"regulator": regulator, "tax_office": tax_office}, candidate_both, z_request
)

tax_on_delivered = step_two_leaders.rewards["harvester:0"][0] - reward_quota_only
assert np.isclose(tax_on_delivered, -0.5 * 0.2 * delivered)
assert abs(tax_on_delivered) < 0.075  # smaller than the tax on the request
print("tax residual with two leaders:", tax_on_delivered)

**A limit of the optimizer.** The sequential behaviour of several leaders is a property of the environment, and it holds for hand stepping and for tests. `ESOptimizer` currently searches the mechanisms of the **first** leader configuration only, so the second leader of a real bilevel run would never receive a candidate action. The next cell shows it: the search space of a two-leader configuration lists only the first leader's mechanism. In an experiment, put the mechanisms you want to optimize together in one leader and rely on the parallel composition of this section.

In [ ]:
two_leader_config = (
    ESConfig()
    .training(episodes=1, sigma=0.15, mean_lr=0.1)
    .agents((regulator, tax_office))
    .debugging(seed=0)
)
assert ESOptimizer(two_leader_config).parameter_names == ["quota"]

# 14. Wiring is dependency injection

Avoid benchmark-specific code inside a generic mechanism. The mechanism asks for things by role, and the configuration of the benchmark says where they are:

- `obs_map` maps a role the mechanism needs (`"resource_level"`) to the state entry of this benchmark;
- `acts_on` names the agents and the mechanism that the intervention targets;
- `mdp.params` carries the constants, such as `K`, that the benchmark chooses to publish.

A mechanism that is wired incompletely fails loudly, at the first step, with a message that names the missing piece. A mechanism wired to a differently named stock works unchanged. The next cell shows both: the tax asks for a role that was not given, and then runs on a benchmark that stores its resource under the name `biomass`.

The helper `scarcity_fixture` builds the one-step state of the analytic test of Section 16: a stock of 200 out of 1000, one harvester that requests \(u=0.4\), and a base reward of 1.

In [ ]:
U_REQUEST = 0.4
# Raw policy output that requests the fraction 0.4: z = T * logit(0.4).
Z_ANALYTIC = ACTION_TEMPERATURE * np.log(U_REQUEST / (1.0 - U_REQUEST))


def scarcity_fixture(state_key):
    """One-step state: stock 200 of K=1000 under `state_key`, request 0.4, reward 1."""
    return MDPState(
        params={"K": 1_000.0},
        state={state_key: 200.0},
        actions={
            "regulator": {"scarcity_tax": np.array([0.5])},
            "harvester:0": {HARVEST: np.array([Z_ANALYTIC])},
        },
        rewards={"harvester:0": 1.0},
    )


# A tax wired without the "resource_level" role fails with a message that names it.
unwired = ScarcityTax(
    id="scarcity_tax", action_space=UNIT, acts_on=(FOLLOWER, HARVEST)
).build("regulator")
try:
    unwired(scarcity_fixture(STOCK), np.array([0.5]))
except ValueError as error:
    print("ValueError:", error)
else:
    raise AssertionError("a tax without obs_map must be rejected")

# The same mechanism, wired to a benchmark that calls its resource "biomass".
on_biomass = ScarcityTax(
    id="scarcity_tax",
    action_space=UNIT,
    acts_on=(FOLLOWER, HARVEST),
    obs_map={"resource_level": "biomass"},
).build("regulator")
residual = on_biomass(scarcity_fixture("biomass"), np.array([0.5]))
assert np.isclose(residual.rewards["harvester:0"][0], -0.5 * 0.8 * 0.4)

# 15. Test the benchmark equation before RL

Do not start with APPO or ES. First test the transition as a pure mathematical function. The first cell evaluates the function of Section 7.1 at the point of the hand calculation below.

In [ ]:
x_next, x_delivered = logistic_step(800.0, 50.0, r=0.3, K=1_000.0)

x_next

Manual verification:

\[
G(800)
=
0.3(800)(1-0.8)
=
48.
\]

Therefore:

\[
x_{t+1}
=
800+48-50
=
798.
\]

The boundaries matter as much as the typical case: the stock never leaves \([0,K]\), and the delivered harvest never exceeds what the resource holds.

In [ ]:
assert np.isclose(x_next, 798.0)
assert x_delivered == 50.0

# An extraction far above the stock empties it, and delivers only what was there.
empty, delivered_all = logistic_step(100.0, 10_000.0, r=0.3, K=1_000.0)
assert empty == 0.0
assert np.isclose(delivered_all, 100.0 + 0.3 * 100.0 * 0.9)

# Without extraction the stock grows, and it never exceeds the capacity.
assert logistic_step(800.0, 0.0, r=0.3, K=1_000.0)[0] > 800.0
assert logistic_step(1_000.0, 0.0, r=0.3, K=1_000.0)[0] == 1_000.0

# 16. Test the custom mechanism analytically

Choose a tax rate of 0.5 and a resource level of 0.2 (a stock of 200 out of 1000), so that the scarcity is 0.8. The harvester requests the extraction fraction 0.4 and the base reward is 1.0.

Expected:

\[
r^\*
=
1.0
-
0.5(0.8)(0.4)
=
0.84.
\]

The cell below runs the real mechanism on the fixture of Section 14 and adds its residual to the base reward with `MDPState.add`, which is what the environment does at every step. The expected values are the hand values above.

In [ ]:
analytic = scarcity_fixture(STOCK)
tax_residual = tax_mechanism(analytic, np.array([0.5]))
shaped = analytic.add(tax_residual)

assert np.isclose(tax_residual.rewards["harvester:0"][0], -0.16)
assert np.isclose(shaped.rewards["harvester:0"][0], 0.84)
shaped.rewards["harvester:0"]

Every mechanism should have an analytical test like this before it is used in a distributed RL run.

# 17. Test the decoding of the optimizer coordinate

The optimizer only ever sees a coordinate in \([0,1]\), and `decode` is the only map from it to the semantic value. For every optimized mechanism, test three properties of `decode`:

1. **The semantic value is right.** The tax rate is the coordinate times `max_rate`.
2. **Out-of-range input is clipped.** Whatever arrives, the decoded value lies in the semantic bounds.
3. **It is pure.** The same input gives the same output, and the state is left unchanged. The environment hands `decode` the raw action as it arrived, so `decode` does not need to be idempotent.

The checks run on the raw action, with the shape the optimizer provides.

In [ ]:
def snapshot(mdp):
    """Plain-Python copy of every trajectory of an MDPState, for before/after checks."""

    def plain(value):
        if isinstance(value, dict):
            return {key: plain(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [plain(item) for item in value]
        return np.asarray(value).tolist()

    names = ("state", "actions", "raw_actions", "rewards", "obs")
    return {name: plain(getattr(mdp, name).data) for name in names}


state = scarcity_fixture(STOCK)  # `decode` ignores the state; any state will do
half_rate = ScarcityTax(id="scarcity_tax", action_space=UNIT, max_rate=0.6).build(
    "regulator"
)

# 1. The semantic value follows the coordinate.
assert np.isclose(half_rate.decode(state, np.array([1.0])), 0.6)
assert np.isclose(half_rate.decode(state, np.array([0.5])), 0.3)

# 2. Out-of-range coordinates are clipped to [0, max_rate].
assert half_rate.decode(state, np.array([-0.3])) == 0.0
assert np.isclose(half_rate.decode(state, np.array([7.0])), 0.6)

# 3. The decoding is pure for the tax and for the shipped quota: same input, same
#    output, and the state is unchanged.
quota_mechanism = quota.build("regulator")
before = snapshot(state)
for coordinate in (-0.3, 0.0, 0.25, 0.8, 1.0, 7.0):
    for mechanism in (tax_mechanism, quota_mechanism):
        first = mechanism.decode(state, np.array([coordinate]))
        assert mechanism.decode(state, np.array([coordinate])) == first
        assert 0.0 <= first <= 1.0
assert snapshot(state) == before

# An invalid fixed parameter is rejected when the mechanism is built.
try:
    ScarcityTax(id="scarcity_tax", action_space=UNIT, max_rate=1.5).build("regulator")
except ValueError as error:
    print("ValueError:", error)
else:
    raise AssertionError("max_rate above 1 must be rejected")

# 18. Test one complete regulated step

For a deterministic fixture, fix the candidate that the regulator publishes and supply a known raw policy action. Then compute from the formulas the delivered action, the transition, the base reward, the mechanism reward and the observation, including the slot that the mechanism fills, and compare each intermediate value with what the environment produced. This is much easier to debug than checking only the final episode return.

The cell below does it for the regulator that owns the quota and the tax, with \(q=0.82\), \(\lambda=0.5\) and a request of 0.75. The expected values are computed in the cell from the equations of this notebook: the quota formula of Section 9, the logistic growth written out in full, and the reward and the tax of Sections 6 and 10. The only numbers taken from the environment are the ones being checked. The tolerance is tight (a relative \(10^{-4}\)), because the two sides compute the same quantity.

In [ ]:
env_full, step_full = run_step({"regulator": both}, candidate_both, z_request)

# Delivered action: the quota formula applied to the request of 0.75 (Section 9).
allowed = allowed_fraction(0.8, 0.82)
fraction = expected_delivered(0.75, allowed)
assert np.isclose(step_full.actions["harvester:0"][HARVEST][0], fraction, rtol=1e-4)

# Transition: growth 0.3 * 800 * (1 - 0.8) = 48, minus the two harvesters, who take
# fraction * H_max = fraction * 20 units each.
harvest = 2 * fraction * 20.0
stock = 800.0 + 0.3 * 800.0 * (1.0 - 0.8) - harvest
assert np.isclose(step_full.state[STOCK][1], stock, rtol=1e-4)

# Reward: the base reward is the delivered fraction; the tax prices the request 0.75.
tax_reward = -0.5 * (1.0 - 0.8) * 0.75
assert np.isclose(tax_reward, -0.075)
for aid in followers:
    assert np.isclose(step_full.rewards[aid][0], fraction + tax_reward, rtol=1e-4)

# Observation of the new step: the normalized stock, and the tax rate in slot 1.
assert np.allclose(step_full.obs["harvester:0"][1], [stock / 1_000.0, 0.5], rtol=1e-4)

# What the logger recorded is what the formulas give (Section 21 compares the agents).
logged = env_full.logger.peek()
assert np.isclose(logged.realized_harvest_total[-1], harvest, rtol=1e-4)
assert np.isclose(logged.stock_norm_last[-1], stock / 1_000.0, rtol=1e-4)
print(
    f"delivered {fraction:.4f}, reward {fraction + tax_reward:.4f}, stock {stock:.2f}"
)

# 19. Action normalization

The raw policy output \(z\in\mathbb R\) becomes a fraction through

\[
a=\operatorname{sigmoid}(z/T)
\]

with

\[
T=4
\]

(the constant `ACTION_TEMPERATURE` of `core.utils`, which the library describes as a heuristic scale, not a value taken from the literature). The step of the environment does **not** decode before the leaders act, so the two sides of the interface see different things:

- a **leader's mechanism** reads the raw \(z\) that the policy emitted, because the leader acts first. A mechanism that targets followers therefore applies the sigmoid itself, as `Quota` and the tax of Section 10 do. A leader lowers an action by adding a residual to that raw value;
- the **follower's own mechanism** then decodes the raw value that is left, \(z'\), to \(a=\operatorname{sigmoid}(z'/T)\in(0,1)\), and everything after the leaders (the reward, the transition) reads \(a\).

If your benchmark needs another physical range, add an explicit conversion in the follower's `decode`. Do not make mechanisms guess whether a number is a logit, a normalized fraction or a physical unit: the follower mechanism documents what it decodes, and a leader's mechanism documents that it reads the raw output. The next cell checks both sides on the step of Section 18: the request was \(z\), the quota lowered it to \(z'\), and the delivered fraction is the sigmoid of \(z'\).

In [ ]:
lowered_raw = float(
    step_full.raw_actions["harvester:0"][HARVEST][0][0]
)  # z' after the leaders
delivered_to_world = step_full.actions["harvester:0"][HARVEST][0]  # decoded from z'

assert lowered_raw < z_request  # the quota lowered the raw request
assert np.isclose(delivered_to_world, sigmoid(lowered_raw / ACTION_TEMPERATURE))
assert delivered_to_world < sigmoid(
    z_request / ACTION_TEMPERATURE
)  # below the 0.75 asked

# The quota hands back the logit of the fraction it delivers (the formula of Section 9).
assert np.isclose(
    lowered_raw, ACTION_TEMPERATURE * np.log(fraction / (1.0 - fraction)), rtol=1e-4
)

# 20. Observation spaces must include mechanism augmentation

If:

\[
o_t\in\mathbb R^{d_o}
\]

and a mechanism fills \(d_m\) entries of the observation, then the observation of the agent is

\[
o_t^\*\in\mathbb R^{d_o+d_m}.
\]

The mechanisms write into slots that the agent **reserved**. The declared Gymnasium space must therefore already include those slots, and the agent's `observation` must leave them at zero. The harvester of this notebook reserves one slot for the tax rate, so its space has \(d_o+d_m=1+1\) entries.

A mechanism that needs more room says so loudly. The social-influence mechanism writes the peers' last delivered actions at an `obs_offset`, and it needs

\[
d_m=(N-1)d_a
\]

reserved entries, for \(N\) agents with actions of size \(d_a\). Pointed at an offset that lies outside the two entries of the harvester, it raises a `ValueError` instead of writing out of bounds. The cell checks that the adapter's observations belong to the declared space, and shows that error.

In [ ]:
context = candidate(**candidate_both)
with stand_in_ray_get():
    adapter = RLlibMultiAgentEnvAdapter(
        build_env(context, {"regulator": both}, horizon=5)
    )
    obs, _ = adapter.reset()
    for agent_id, value in obs.items():
        assert adapter.observation_spaces[agent_id].contains(value)
    obs, *_ = adapter.step(
        {aid: {HARVEST: np.array([0.0], np.float32)} for aid in followers}
    )
    for agent_id, value in obs.items():
        assert adapter.observation_spaces[agent_id].contains(value)
        assert value.shape == (2,)

# Social influence needs reserved slots that the harvester does not declare.
social = SocialInfluence(id="social", acts_on=(FOLLOWER, HARVEST), obs_offset=2)
leader_with_social = AgentConfig(
    id="regulator", policy_id="regulator_policy", mechanisms=(quota, social)
)
try:
    run_step(
        {"regulator": leader_with_social},
        {"quota": np.array([0.5], np.float32), "social": np.empty(0, np.float32)},
        0.0,
    )
except ValueError as error:
    print("ValueError:", error)
else:
    raise AssertionError("an offset outside the observation must be rejected")

# 21. Context publication

A regulated step should publish the values actually used by the system. The candidates and the metrics are the two channels that carry them.

The World and the environments exchange candidates. The base `RegulatorEnv` publishes the population of candidates to the World as `MechanismContext` objects whose status is `published` and whose `env_id` is `None`. An environment of the inner level fetches its candidate at reset, which the stand-in World shows by emptying its list of answers. The `done` context that carries the fitness back is not published by the base class. It is published by the `reward` method of your regulator environment (Section 23).

The metrics are the values that the regulator's fitness is computed from. The environment's logger records, at each step, the stock after the transition and the mean reward of the step. They must equal what the agents actually received.

If what is published disagrees with what drove the transition or the learner, debugging and reproducibility become unreliable. The cell runs several steps through the adapter. It checks the candidate through the observation, where the tax rate that the candidate carries appears in slot 1. At every step it compares what the agents received, the observation and the rewards that the adapter returns, with what the logger recorded.

In [ ]:
HORIZON = 5
with stand_in_ray_get():
    env_log = build_env(
        candidate(**candidate_both), {"regulator": both}, horizon=HORIZON
    )
    adapter = RLlibMultiAgentEnvAdapter(env_log)
    assert len(env_log.world.answers) == 1  # the candidate waits in the World
    obs, _ = adapter.reset()
    assert env_log.world.answers == []  # the environment fetched it at reset
    assert np.isclose(
        obs["harvester:0"][1], 0.5
    )  # its tax rate reached the observation

    for _ in range(HORIZON):
        obs, rewards, *_ = adapter.step(
            {aid: {HARVEST: np.array([1.0], np.float32)} for aid in followers}
        )
        logged = env_log.logger.peek()
        # The stock the agents observe is the stock the logger recorded.
        assert np.isclose(obs["harvester:0"][0], logged.stock_norm_last[-1], rtol=1e-5)
        # The mean of the rewards the agents received is the logged mean reward.
        assert np.isclose(
            np.mean(list(rewards.values())), logged.reward_mean[-1], rtol=1e-5
        )

episode_metrics = env_log.logger.reduce()
assert np.isclose(episode_metrics.stock_norm_last, obs["harvester:0"][0], rtol=1e-5)
episode_metrics.realized_harvest_total

# 22. Reproducibility checklist

Record the benchmark constants (`r`, `K`, `H_max`, the initial stock and its noise), the mechanism parameters and fixed configuration fields (`max_rate`, the widths of the quota), and the action semantics, that is the id and the decoding of every mechanism. Record the environment seed, the policy seed, the evaluation seeds, the horizon, the number of agents, the stochastic-noise model (here only the optional noise on the initial stock) and the optimizer seed.

Validate in this order: the deterministic transition, the deterministic mechanism transform, the deterministic full environment step, a repeated seeded episode, the inner RL, and the outer optimization. Steps 1 to 3 are Sections 15, 16 and 18. The next cell does step 4. This benchmark is deterministic unless the initial stock is noisy, and a test of seeding is only meaningful if there is a randomness to seed. The cell therefore sets `x_init_sigma=0.05`, so the initial stock is drawn from the seeded generator of the environment. Two environments built with the same seed and the same candidate must produce the same trajectory. A different seed must give a different initial stock, and the episode must end at the horizon with a truncation flag.

For the runs of steps 5 and 6, the Python hash seed also matters, because it changes the iteration order of sets of strings, and RLlib iterates over a set of agent identifiers. Dictionaries keep their insertion order, so they are not affected. Set `PYTHONHASHSEED` before the interpreter starts, or call `ensure_hash_seed()` from `core.config.hash_seed` at the top of a script entry point. A notebook kernel is already running, so set it in the environment that launches Jupyter.

In [ ]:
def seeded_episode(seed):
    """Trace of one episode through the adapter, with a fixed action sequence."""
    with stand_in_ray_get():
        env = build_env(
            candidate(**candidate_both),
            {"regulator": both},
            horizon=5,
            seed=seed,
            ecology={"x_init_sigma": 0.05},
        )
        adapter = RLlibMultiAgentEnvAdapter(env)
        obs, _ = adapter.reset()
        trace = [(obs["harvester:0"].copy(), 0.0)]
        for step_index in range(5):
            z = np.float32(1.0 - 0.5 * step_index)
            obs, rewards, _, truncateds, _ = adapter.step(
                {aid: {HARVEST: np.array([z], np.float32)} for aid in followers}
            )
            trace.append((obs["harvester:0"].copy(), rewards["harvester:0"]))
    return trace, truncateds


first_trace, first_truncated = seeded_episode(seed=3)
second_trace, _ = seeded_episode(seed=3)
other_trace, _ = seeded_episode(seed=4)

# The initial stock is really drawn: it differs from the default 0.8 of the other cells.
assert not np.isclose(first_trace[0][0][0], 0.8)

# The same seed gives the same trajectory, the first observation included.
for (obs_a, reward_a), (obs_b, reward_b) in zip(first_trace, second_trace):
    assert np.array_equal(obs_a, obs_b)
    assert reward_a == reward_b

# Another seed gives another initial stock.
assert not np.array_equal(first_trace[0][0], other_trace[0][0])
assert first_truncated["__all__"]  # the episode ends at the horizon

# 23. Run the benchmark in a bilevel experiment

Only after the checks above should the benchmark be used in an expensive bilevel run. The run below is tiny on purpose: two ES generations, two inner training iterations, two candidates and an episode of 20 steps. It checks that every stage works together, and it does not produce a meaningful optimum.

Three pieces connect the benchmark to the bilevel loop.

**The regulator environment.** `RegulatorEnv` publishes the population of candidates to the World, lets the inner optimizer train against them, and then calls your `reward(metrics)`. The method turns the inner metrics into one fitness per candidate. The base class publishes only the candidates, so your `reward` also appends, for each candidate, a `done` `MechanismContext` whose `metrics` is a `ContextSchema` payload. The metrics tree is `metrics.train.rollout.by_mechanism[id].by_seed[...].by_episode[...]`, whose leaves are the fields of the `CommonPoolMetricSchema` of Section 5.2, one entry per inner iteration, with `None` for a gap.

**The objective.** The fitness below adds the harvest delivered over the episode, as a fraction of the most the harvesters could have taken, and the normalized stock left at the end. It rewards yield and sustainability together. This weighting is a simple heuristic chosen for the tutorial. It is not taken from the literature, and a real study must justify its objective.

**The configuration.** `BilevelConfig` takes the regulator (an ES over the leader's mechanisms), the society (an APPO over the followers) and the reporter. The value that the `env_runners` call passes as `num_envs_per_env_runner` is the number of candidates in the ES population. The configuration multiplies it by the number of training seeds before it gives it to RLlib (one seed here), so each seed evaluates that many candidates. The evolution strategy samples its population in antithetic pairs, so the count must be even, unless it is 1 or `break_symmetry` is set.

The first cell defines the objective and checks it on cases with a known answer.

In [ ]:
INNER_HORIZON = 20
H_MAX = build_env().H_max  # per-harvester extraction capacity of the environment


def episode_objective(harvest_total, stock_norm_last):
    """Normalized yield plus normalized final stock, each in [0, 1]."""
    best_possible_harvest = NUM_HARVESTERS * H_MAX * INNER_HORIZON
    return harvest_total / best_possible_harvest + stock_norm_last


assert episode_objective(0.0, 1.0) == 1.0  # nothing harvested, full stock
assert episode_objective(0.0, 0.0) == 0.0  # collapsed stock, nothing harvested
assert episode_objective(NUM_HARVESTERS * H_MAX * INNER_HORIZON, 0.0) == 1.0

In [ ]:
class FitnessContext(ContextSchema):
    """Fitness of one candidate, carried back to the World."""

    objective_score: float
    mean_yield: float
    mean_final_stock: float


class CommonPoolRegulatorEnv(RegulatorEnv):
    """Outer-loop environment that scores common-pool candidates."""

    def reward(self, metrics):
        fitness = {}
        for (
            mechanism_id,
            mechanism_metrics,
        ) in metrics.train.rollout.by_mechanism.items():
            harvests, stocks = [], []
            for seed_metrics in mechanism_metrics.by_seed.values():
                for episode in seed_metrics.by_episode.values():
                    harvests += [
                        float(v)
                        for v in np.atleast_1d(episode.realized_harvest_total)
                        if v is not None
                    ]
                    stocks += [
                        float(v)
                        for v in np.atleast_1d(episode.stock_norm_last)
                        if v is not None
                    ]
            mean_yield, mean_stock = float(np.mean(harvests)), float(np.mean(stocks))
            fitness[int(mechanism_id)] = (
                episode_objective(mean_yield, mean_stock),
                mean_yield,
                mean_stock,
            )

        scores = np.full(max(fitness) + 1, -np.inf, dtype=np.float32)
        for index, (objective, mean_yield, mean_stock) in fitness.items():
            scores[index] = objective
            payload = FitnessContext(
                objective_score=objective,
                mean_yield=mean_yield,
                mean_final_stock=mean_stock,
            )
            ray.get(
                self.world.append_context.remote(
                    Context(
                        id=None,
                        opt_id=self.opt_id,
                        # `_t` is the step counter of this environment. It has no public
                        # accessor, so it is read directly.
                        step=self._t,
                        env=self.__class__.__name__,
                        payload=MechanismContext(
                            index=index,
                            seed=None,
                            status=MechanismStatus.done,
                            env_id=None,
                            mechanism=None,
                            metrics=payload,
                        ),
                    )
                )
            )
        return scores.tolist()

The `.evaluation` call of the inner level gives only what the library cannot decide for you: the `evaluation_config` and the evaluation seeds. The library forces the evaluation interval and `evaluation_parallel_to_training=False`, and it overrides the evaluation runners, the duration and the custom evaluation function. One setting is mandatory: `evaluation_config["rollout_fragment_length"]` must be given, and the optimizer raises an error without it.

The configuration cell builds the whole experiment as one `BilevelConfig`. The leader owns the quota and the tax (the penalty is left out, so the search has two dimensions), and the society is the two harvesters, given as one `HarvesterConfig` with `count=2`. The inner algorithm settings follow the minimal APPO configuration of `examples/cartpole/debug.py`. The reporter writes CSV files into a temporary directory, so running the notebook leaves nothing behind.

In [ ]:
def build_bilevel(output_dir, *, generations=2, inner_iterations=2, candidates=2):
    leader = AgentConfig(
        id="regulator", policy_id="regulator_policy", mechanisms=(quota, tax)
    )
    society = dataclasses.replace(harvester, count=NUM_HARVESTERS)
    batch = INNER_HORIZON * candidates
    return (
        BilevelConfig()
        .world(world_name="common_pool_world")
        .reporter(config=CSVConfig(project="common_pool", output_dir=output_dir))
        .ray(device="cpu", num_cpus=4, omp_threads=1, logging_level="ERROR")
        .regulator(
            ESConfig()
            .training(sigma=0.15, mean_lr=0.1, episodes=generations)
            .agents(leader)
            .environment(horizon=1, env=CommonPoolRegulatorEnv)
            .debugging(seed=0)
        )
        .society(
            APPOptimizerConfig()
            .resources(num_cpus_for_main_process=1)
            .framework(framework="torch")
            .api_stack(
                enable_rl_module_and_learner=True,
                enable_env_runner_and_connector_v2=True,
            )
            .environment(
                env=CommonPoolEnv,
                env_config={"ecology_cfg": {}, "seed": 0},
                horizon=INNER_HORIZON,
                disable_env_checking=False,
                schema=CommonPoolMetricSchema,
                queries=(),
            )
            .env_runners(
                num_env_runners=0,
                num_cpus_per_env_runner=1,
                num_gpus_per_env_runner=0,
                num_envs_per_env_runner=candidates,
                rollout_fragment_length=INNER_HORIZON,
                batch_mode="truncate_episodes",
            )
            .learners(num_learners=0, num_gpus_per_learner=0)
            .callbacks(
                on_episode_created=tag_episode_with_env_idx,
                on_episode_end=log_and_report_episode_metrics,
            )
            .training(
                episodes=inner_iterations,
                vtrace=True,
                circular_buffer_num_batches=2,
                circular_buffer_iterations_per_batch=1,
                broadcast_interval=5,
                timeout_s_sampler_manager=300,
                timeout_s_aggregator_manager=300,
                gamma=0.99,
                lr=0.001,
                train_batch_size_per_learner=batch,
                minibatch_size=batch,
                entropy_coeff=0.01,
                grad_clip=40.0,
            )
            .evaluation(
                evaluation_config={
                    "explore": False,
                    "rollout_fragment_length": INNER_HORIZON,
                    "batch_mode": "complete_episodes",
                },
                base_seed=42,
                num_seeds=1,
            )
            .agents(society)
            .fault_tolerance(restart_failed_env_runners=False)
            .debugging(seed=42, num_seeds=1)
            .reporting(
                min_time_s_per_iteration=0,
                min_sample_timesteps_per_iteration=0,
                min_train_timesteps_per_iteration=0,
                queries=(),
            )
        )
    )


with tempfile.TemporaryDirectory() as scratch_dir:
    bilevel_cfg = build_bilevel(scratch_dir)
assert isinstance(bilevel_cfg, BilevelConfig)

`build_optimizer()` starts Ray and builds the World and both optimizers, and `train()` runs the outer loop. Call `ray.shutdown()` before building, so that a re-run of the notebook does not inherit a stale runtime, and again at the end, even after a failure. The library does not configure logging, so the cell turns on the `INFO` level of the root logger: the evolution strategy then prints several `INFO` lines per generation (the start of the generation, the population before the update, the parameter gradients, the mean after the update, a summary and a convergence check), interleaved with the lines of the inner optimizer. The same level also turns on the messages of Ray and RLlib.

`train()` returns the summary of the outer optimizer. Its `best_mechanism` is the best candidate as a vector of coordinates in \([0,1]\), in the order of `parameter_names` (Section 12). Decoding the coordinate of the tax with the mechanism's own `decode` gives the rate the regulator found. With two generations the value is close to the starting point and carries no scientific meaning.

In [ ]:
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

ray.shutdown()
try:
    with tempfile.TemporaryDirectory() as scratch_dir:
        optimizer = build_bilevel(scratch_dir).build_optimizer()
        summary = optimizer.train()
finally:
    ray.shutdown()
assert not ray.is_initialized()

best = dict(zip(searched.parameter_names, summary["best_mechanism"]))
assert summary["episodes"] == 2
assert set(best) == {"quota", "scarcity_tax"}
assert all(0.0 <= value <= 1.0 for value in best.values())
assert np.isfinite(summary["best_fitness"])

tax_rate = tax_mechanism.decode(MDPState(), np.array([best["scarcity_tax"]]))
print(f"best fitness {summary['best_fitness']:.4f}")
print(f"quota {best['quota']:.3f}, tax rate {tax_rate:.3f}")

# 24. Minimal test suite for a new benchmark

The tests of the repository show the patterns to copy. `tests/envs/conftest.py` holds a toy environment and a scripted World, and an identity `ray.get`, so environments are tested without Ray. `tests/integration/conftest.py` builds a real environment behind the real RLlib adapter, with a scripted World and a `candidate_context` factory. These are the same stand-ins as the ones of Section 5.1.

For a new benchmark, write at least these tests, each named after the behaviour it fixes. The names below are the real ones of the repository, and the cell at the end of this section checks that each of them still exists.

**Benchmark**, in `tests/examples/test_fishery_env_dynamics.py`

- `test_deterministic_reset_starts_at_the_configured_stock`: the reset state and the constants in `params`;
- `test_stochastic_reset_is_reproducible_and_clipped_to_capacity`: the seeded repeat of Section 22;
- `test_one_step_of_the_schaefer_equation_by_hand`: the transition against a hand calculation;
- `test_stock_never_goes_negative_under_overfishing` and `test_the_catch_never_exceeds_the_stock_it_was_taken_from`: the transition boundaries;
- `test_reward_is_the_harvest_fraction_of_the_step`: the base reward, on the decoded action;
- `test_first_observation_is_the_normalised_stock_and_usage`: the observation;
- `test_the_outcome_does_not_depend_on_the_order_of_the_fishers`: symmetry between agents;
- `test_an_env_without_a_schema_steps_without_logging` and `test_the_catch_of_every_step_follows_the_stock_of_that_step`: the metrics agree with the state.

**Mechanism**, in `tests/mechanism/algorithms/test_quota.py`

- `test_decode_clips_the_quota_to_the_unit_interval` and `test_call_writes_the_decoded_quota_back`: the decoding of the coordinate;
- `test_apply_requires_a_target` and `test_apply_requires_the_resource_level`: incomplete wiring raises `ValueError`;
- `test_delivery_follows_the_allowed_fraction`: the analytic value, as in Section 16;
- `test_request_below_the_allowance_is_left_unchanged` and `test_only_agents_of_the_targeted_type_are_capped`: what the mechanism must not touch;
- `test_config_forwards_the_widths`: the fixed parameters reach the mechanism.

**Composition and episode**, in `tests/integration/test_fishery_regulator_composition.py`

- `test_each_residual_is_active_in_the_episode` and `test_reward_matches_the_formulas_of_the_three_mechanisms`: the compound reward equals the sum of the residuals;
- `test_without_a_candidate_the_reward_is_the_requested_harvest`: the environment also runs without a leader;
- `test_the_episode_is_truncated_at_the_horizon`.

The repository has no test of the order independence of Section 13, so a benchmark that composes several mechanisms should add one, as the cell of that section does.

Only after these pass should the benchmark be used in an expensive bilevel run. The next cell checks that the test modules this section points to still exist, and that each test name it quotes is still defined, so that the pointers cannot rot unnoticed.

In [ ]:
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
pattern_modules = {
    "tests/envs/conftest.py": ["class ScriptedWorld", "def identity_ray_get("],
    "tests/integration/conftest.py": ["class ScriptedWorld", "def candidate_context("],
    "tests/examples/test_fishery_env_dynamics.py": [
        "def test_deterministic_reset_starts_at_the_configured_stock(",
        "def test_stochastic_reset_is_reproducible_and_clipped_to_capacity(",
        "def test_one_step_of_the_schaefer_equation_by_hand(",
        "def test_stock_never_goes_negative_under_overfishing(",
        "def test_the_catch_never_exceeds_the_stock_it_was_taken_from(",
        "def test_reward_is_the_harvest_fraction_of_the_step(",
        "def test_first_observation_is_the_normalised_stock_and_usage(",
        "def test_the_outcome_does_not_depend_on_the_order_of_the_fishers(",
        "def test_an_env_without_a_schema_steps_without_logging(",
        "def test_the_catch_of_every_step_follows_the_stock_of_that_step(",
    ],
    "tests/mechanism/algorithms/test_quota.py": [
        "def test_decode_clips_the_quota_to_the_unit_interval(",
        "def test_call_writes_the_decoded_quota_back(",
        "def test_apply_requires_a_target(",
        "def test_apply_requires_the_resource_level(",
        "def test_delivery_follows_the_allowed_fraction(",
        "def test_request_below_the_allowance_is_left_unchanged(",
        "def test_only_agents_of_the_targeted_type_are_capped(",
        "def test_config_forwards_the_widths(",
    ],
    "tests/mechanism/algorithms/test_subsidy.py": [],
    "tests/mechanism/algorithms/test_penalty.py": [],
    "tests/mechanism/algorithms/test_social_influence.py": [],
    "tests/integration/test_fishery_regulator_composition.py": [
        "def test_each_residual_is_active_in_the_episode(",
        "def test_reward_matches_the_formulas_of_the_three_mechanisms(",
        "def test_without_a_candidate_the_reward_is_the_requested_harvest(",
        "def test_the_episode_is_truncated_at_the_horizon(",
    ],
}

# Every file exists, and every name quoted in the text is still defined in its file.
missing = []
for name, needles in pattern_modules.items():
    path = REPO_ROOT / name
    if not path.is_file():
        missing.append(name)
        continue
    text = path.read_text()
    missing += [f"{name}: {needle}" for needle in needles if needle not in text]
assert not missing, missing

# 25. Pull-request checklist for a new benchmark

## Theory

- state is mathematically defined;
- action semantics are mathematically defined;
- transition equation is documented;
- base reward is documented;
- observation is documented;
- mechanism intervention point is documented.

## Code

- a `@reset` method and a `@transition` method (one of each, and no other hooks);
- `observation` and `reward` methods on the `Agent` subclass, with the observation slots reserved for mechanisms;
- an `AgentConfig` subclass, with follower ids of the form `"<agent>:<i>"`;
- a follower `Mechanism` and `MechanismConfig` for every action component, with the declared `action_space`;
- `observation_space` that includes the slots the mechanisms fill;
- a `MetricSchema` with a reduction declared for every field, and `logger.push` calls in the transition;
- the constants that generic mechanisms need published in `params`, and the state keys documented for `obs_map`;
- for a learned mechanism, an `action_space` that is a `Box` in \([0,1]\), a `decode` that clips and does not modify the state, and `empty_action_space()` for a fixed rule;
- for a bilevel experiment, a `RegulatorEnv` subclass with a documented objective and a `ContextSchema` payload.

## Tests

- equations tested numerically;
- mechanism tested independently, including `decode` and the wiring errors;
- composition tested;
- deterministic end-to-end step tested;
- seeded reproducibility tested;
- the tutorial notebook, or an equivalent smoke run, executes end to end.

This order keeps the code tied to a scientific model rather than letting the class hierarchy define the experiment.